# Notebook 1: Crystal Structures — Representation & Tools

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

In this notebook you will learn to:
- Represent crystal structures in Python using **pymatgen**
- Create structures from scratch (lattice + basis)
- Load structures from CIF files
- Inspect lattice parameters, compositions, and coordinates
- Visualize unit cells in 3D

> **Prerequisites:** Run `00_setup.ipynb` first to install dependencies.

In [ ]:
# --- Run once per Colab session (skip if you already ran 00_setup.ipynb) ---
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

# Clone repo if needed
if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')

# Install all tutorial dependencies from the shared requirements file
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', f'{PROJECT_DIR}/requirements-colab.txt'])
os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')

---
## 1.1 What is a crystal structure?

A crystal structure is defined by:
- A **lattice** — the periodic box described by three vectors **a**, **b**, **c** (or equivalently, lengths *a, b, c* and angles *α, β, γ*)
- A **basis** — the atoms inside one unit cell, given by their species and fractional coordinates

In pymatgen, these are combined into a `Structure` object.

In [ ]:
from pymatgen.core import Structure, Lattice
import numpy as np

### Creating a structure from scratch

Let's build a simple cubic NaCl (rock salt) structure.
The conventional cell has lattice parameter *a* ≈ 5.64 Å.

In [ ]:
# Define the lattice: cubic with a = 5.64 Å
lattice = Lattice.cubic(5.64)

# Define species and fractional coordinates
species = ['Na', 'Na', 'Na', 'Na', 'Cl', 'Cl', 'Cl', 'Cl']
frac_coords = [
    [0.0, 0.0, 0.0],  # Na
    [0.5, 0.5, 0.0],  # Na
    [0.5, 0.0, 0.5],  # Na
    [0.0, 0.5, 0.5],  # Na
    [0.5, 0.0, 0.0],  # Cl
    [0.0, 0.5, 0.0],  # Cl
    [0.0, 0.0, 0.5],  # Cl
    [0.5, 0.5, 0.5],  # Cl
]

nacl = Structure(lattice, species, frac_coords)
print(nacl)

### Inspecting a structure

pymatgen structures expose all the information you need:

In [ ]:
print(f'Formula:           {nacl.composition.reduced_formula}')
print(f'Number of atoms:   {nacl.num_sites}')
print(f'Volume:            {nacl.volume:.2f} Å³')
print(f'Density:           {nacl.density:.3f} g/cm³')
print()
print('Lattice parameters:')
print(f'  a={nacl.lattice.a:.2f} Å, b={nacl.lattice.b:.2f} Å, c={nacl.lattice.c:.2f} Å')
print(f'  α={nacl.lattice.alpha:.1f}°, β={nacl.lattice.beta:.1f}°, γ={nacl.lattice.gamma:.1f}°')
print()
print('Fractional coordinates:')
for site in nacl:
    print(f'  {site.species_string:>4s}  {site.frac_coords}')

---
## 1.2 Loading structures from CIF files

CIF (Crystallographic Information File) is the standard format for crystal structures.
SCIGEN's training data contains CIF strings in CSV files. Let's load one.

In [ ]:
import pandas as pd
import os

# Load the SCIGEN training data (MP-20 dataset)
PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')
df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'mp_20', 'train.csv'))
print(f'Dataset: {len(df)} structures')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Look at the first few entries
df[['material_id', 'pretty_formula', 'formation_energy_per_atom',
    'e_above_hull', 'spacegroup.number']].head(10)

In [ ]:
# Parse a CIF string into a pymatgen Structure
from pymatgen.core import Structure

row = df.iloc[0]
struct = Structure.from_str(row['cif'], fmt='cif')

print(f'Material ID: {row["material_id"]}')
print(f'Formula:     {row["pretty_formula"]}')
print(f'Atoms/cell:  {struct.num_sites}')
print(f'Space group: {row["spacegroup.number"]}')
print()
print(struct)

---
## 1.3 Saving and exporting structures

You can write structures to CIF, POSCAR, or other formats:

In [ ]:
# Save to CIF
cif_string = struct.to(fmt='cif')
print(cif_string[:500])

---
## 1.4 Visualizing crystal structures

Let's visualize a unit cell using matplotlib. We'll plot atoms as spheres
colored by element, and draw the unit cell box.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import to_rgba

# Simple color map for common elements
ELEMENT_COLORS = {
    'H': 'white', 'Li': 'purple', 'C': 'black', 'N': 'blue', 'O': 'red',
    'Na': 'violet', 'Mg': 'green', 'Al': 'gray', 'Si': 'gold',
    'S': 'yellow', 'Cl': 'lime', 'K': 'purple', 'Ca': 'green',
    'Ti': 'silver', 'Mn': 'orchid', 'Fe': 'orange', 'Co': 'dodgerblue',
    'Ni': 'forestgreen', 'Cu': 'brown', 'Zn': 'slategray',
}

def plot_structure(struct, title=None, ax=None):
    """Plot a pymatgen Structure in 3D."""
    if ax is None:
        fig = plt.figure(figsize=(6, 6))
        ax = fig.add_subplot(111, projection='3d')

    # Plot atoms
    cart_coords = struct.cart_coords
    for site in struct:
        c = ELEMENT_COLORS.get(site.species_string, 'gray')
        ax.scatter(*site.coords, s=200, c=c, edgecolors='black', linewidths=0.5,
                   depthshade=True, label=site.species_string)

    # Draw unit cell edges
    lattice_matrix = struct.lattice.matrix
    origin = np.array([0, 0, 0])
    a, b, c = lattice_matrix
    corners = [origin, a, b, c, a+b, a+c, b+c, a+b+c]
    edges = [
        (origin, a), (origin, b), (origin, c),
        (a, a+b), (a, a+c), (b, a+b), (b, b+c),
        (c, a+c), (c, b+c), (a+b, a+b+c), (a+c, a+b+c), (b+c, a+b+c)
    ]
    for start, end in edges:
        ax.plot(*zip(start, end), 'k-', alpha=0.3, linewidth=0.8)

    # De-duplicate legend entries
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc='upper left', fontsize=9)

    if title:
        ax.set_title(title, fontsize=12)
    ax.set_xlabel('x (Å)')
    ax.set_ylabel('y (Å)')
    ax.set_zlabel('z (Å)')
    return ax

# Visualize NaCl
plot_structure(nacl, title='NaCl (rock salt)')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the structure we loaded from the dataset
plot_structure(struct, title=f'{row["pretty_formula"]} ({row["material_id"]})')
plt.tight_layout()
plt.show()

---
## 1.5 The kagome lattice — our running example

The **kagome lattice** is a pattern of corner-sharing triangles that appears in many
frustrated magnets. It is one of the key structural constraints that SCIGEN can target.

Let's build a simple kagome lattice to see what it looks like.

In [ ]:
# Build a simple kagome lattice with Mn atoms
# Hexagonal cell: a = b, gamma = 120°
a_lat = 5.0  # Å
lattice_kag = Lattice.hexagonal(a_lat, 6.0)  # a=b=5.0, c=6.0, gamma=120°

# Kagome sites: 3 atoms per unit cell at specific fractional positions
kagome_frac = [
    [0.5, 0.0, 0.0],
    [0.0, 0.5, 0.0],
    [0.5, 0.5, 0.0],
]

kagome_struct = Structure(lattice_kag, ['Mn'] * 3, kagome_frac)

print('Kagome lattice with Mn:')
print(f'  a = {lattice_kag.a:.2f} Å, c = {lattice_kag.c:.2f} Å, γ = {lattice_kag.gamma:.0f}°')
print(f'  Atoms per cell: {kagome_struct.num_sites}')
print()
print(kagome_struct)

In [ ]:
# Visualize the kagome lattice
# We'll plot a 2x2 supercell in 2D to see the triangular pattern
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 6))

# Create a 3x3 supercell to see the kagome pattern clearly
supercell = kagome_struct.copy()
supercell.make_supercell([3, 3, 1])

# Plot atom positions (projected onto xy plane)
coords = supercell.cart_coords
ax.scatter(coords[:, 0], coords[:, 1], s=120, c='orchid',
           edgecolors='black', linewidths=0.8, zorder=5)

# Draw bonds (connect atoms within ~0.6*a of each other)
from scipy.spatial.distance import cdist
dists = cdist(coords[:, :2], coords[:, :2])
bond_cutoff = a_lat * 0.6
for i in range(len(coords)):
    for j in range(i+1, len(coords)):
        if dists[i, j] < bond_cutoff:
            ax.plot([coords[i, 0], coords[j, 0]],
                    [coords[i, 1], coords[j, 1]],
                    'k-', linewidth=1.2, alpha=0.5)

ax.set_aspect('equal')
ax.set_title('Kagome lattice — corner-sharing triangles', fontsize=13)
ax.set_xlabel('x (Å)')
ax.set_ylabel('y (Å)')
plt.tight_layout()
plt.show()

The kagome pattern of corner-sharing triangles leads to **geometric frustration**
in magnetic materials — spins on a triangle cannot all be antiparallel to each other.
This frustration gives rise to exotic quantum states like spin liquids.

In Notebook 05, we will use SCIGEN to **generate new crystal structures** that
contain this kagome sublattice.

---
## Exercise

1. Pick a different structure from the training data (`df.iloc[100]`, for example). Parse it, print the formula and number of atoms, and visualize it.
2. Try building a **honeycomb** lattice (2 atoms per cell at [1/3, 2/3, 0] and [2/3, 1/3, 0] in a hexagonal cell). Visualize it as a supercell.

In [ ]:
# Your code here


---
## What's next?

In **Notebook 02**, we'll connect to the **Materials Project** database to explore what known materials exist — and motivate why we need generative models to discover new ones.